In [ ]:
# Import required libraries
import torch
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import time
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
import importlib


In [ ]:

# Add current directory to path to import our modules
if '.' not in sys.path:
    sys.path.append('.')

# Import our modules
from models import ObjectDetectionModel, ObjectDetectionDataModule
from utils import plot_image
from onnx_conversion import export_to_onnx, verify_onnx_model


In [ ]:

# reload modules to get the latest changes
import models
import utils
import onnx_conversion

importlib.reload(models)
importlib.reload(utils)
importlib.reload(onnx_conversion)

from models import ObjectDetectionModel, ObjectDetectionDataModule, LightweightObjectDetectionModel
from utils import plot_image, plot_side_by_side, compare_pytorch_onnx_side_by_side
from onnx_conversion import export_to_onnx, verify_onnx_model


# Step 1: Initialize the model and data module

In [ ]:
import os
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define constants
width = 520
height = 240
image_dir_train = "./Dataset/dataset_syn/images/train"
image_dir_val = "./Dataset/dataset_syn/images/val"

# Set up grid lines
grid_lines = [[0, 0.2, 0.4, 0.6, 0.8, 1], [0, 1]]
grid_lines[0] = [line * width for line in grid_lines[0]]
grid_lines[1] = [line * height for line in grid_lines[1]]
danger_levels = 3

# Training parameters
max_epochs = 10  # Adjust as needed
batch_size = 8
checkpoint_path = "./model_checkpoints"
os.makedirs(checkpoint_path, exist_ok=True)

# Create checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath=checkpoint_path,
    filename='object_detection-{epoch:02d}-{val_loss:.2f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min'
)

# Training flag - set to True if you want to train the model, False to load a saved model
train_model = True

# Initialize our ONNX-compatible model
# model = ObjectDetectionModel(grid_lines, danger_levels, input_shape=(3, height, width))
model = LightweightObjectDetectionModel(grid_lines, danger_levels, input_shape=(3, height, width))

# Initialize the data module
data_module = ObjectDetectionDataModule(
    image_dir_train, 
    image_dir_val, 
    width, 
    height, 
    grid_lines, 
    danger_levels, 
    batch_size=batch_size, 
    device=device
)

# Step 2: Train the model or load from checkpoint

In [ ]:


if train_model:
    # Setup PyTorch Lightning trainer
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        callbacks=[checkpoint_callback],
        check_val_every_n_epoch=1,
        log_every_n_steps=10
    )
    
    # Train the model
    print("Starting model training...")
    trainer.fit(model, data_module)
    
    # Get the best checkpoint path
    best_model_path = checkpoint_callback.best_model_path
    if best_model_path:
        print(f"Best model saved at: {best_model_path}")
        # Load the best model
        if (type(model) == ObjectDetectionModel):
            model = ObjectDetectionModel.load_from_checkpoint(best_model_path, grid_lines=grid_lines, danger_levels=danger_levels)
        if type(model) == LightweightObjectDetectionModel:
            model = LightweightObjectDetectionModel.load_from_checkpoint(best_model_path, grid_lines=grid_lines, danger_levels=danger_levels)
    else:
        print("No checkpoints saved. Using the final model state.")
else:
    # Load model from a predefined checkpoint
    checkpoint_file = "checkpoint_filename.ckpt"  # Replace with your checkpoint filename
    checkpoint_full_path = os.path.join(checkpoint_path, checkpoint_file)
    
    if os.path.exists(checkpoint_full_path):
        print(f"Loading model from checkpoint: {checkpoint_full_path}")
        model = ObjectDetectionModel.load_from_checkpoint(
            checkpoint_full_path, 
            grid_lines=grid_lines, 
            danger_levels=danger_levels
        )
    else:
        print(f"Checkpoint file not found: {checkpoint_full_path}")
        print("Proceeding with untrained model.")


# Step 3: Evaluate the model on validation set

In [ ]:


print("Evaluating model on validation set...")
model.eval()
trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu')
val_results = trainer.validate(model, datamodule=data_module)
print(f"Validation results: {val_results}")


# Step 4: Visualize some predictions on validation data

In [ ]:

data_module.setup()
val_dataloader = data_module.val_dataloader()

# Get a batch of validation data
val_batch = next(iter(val_dataloader))
images, targets = val_batch

# Move the model to the same device as the input data
device = images.device
model = model.to(device)

# Make predictions
model.eval()
with torch.no_grad():
    predictions = model(images)

# Display a few examples side by side
batch_size = images.shape[0]
num_examples = min(3, batch_size)

for i in range(num_examples):
    # Plot ground truth and prediction side by side
    plot_side_by_side(
        image=images[i],
        ground_truth=targets[i],
        prediction=predictions[i],
        grid_lines=grid_lines,
        example_num=i+1
    )
    plt.show()

# Step 5: Convert the PyTorch model to ONNX

In [ ]:
# Move model to CPU, then convert to ONNX
model = model.to('cpu')
dummy_input = torch.randn(1, 3, height, width, device='cpu')

export_to_onnx(model, input_shape=(1, 3, height, width), output_path="./light_object_detection.onnx")

In [ ]:

# Try loading the model to verify it works
import onnxruntime
ort_session = onnxruntime.InferenceSession("object_detection.onnx")

# Test with a sample input
test_input = images[0:1].cpu().numpy()
ort_inputs = {ort_session.get_inputs()[0].name: test_input}
ort_outputs = ort_session.run(None, ort_inputs)

print("Direct ONNX export successful!")

In [ ]:
def run_pytorch_onnx_comparison(model, onnx_path, data_module, num_examples=3):
    """
    Run comparison between PyTorch and ONNX models
    
    Args:
        model: The PyTorch model
        onnx_path: Path to the ONNX model
        data_module: PyTorch Lightning data module
        num_examples: Number of examples to visualize
    """
    # Ensure setup is called
    data_module.setup()
    val_dataloader = data_module.val_dataloader()
    
    # Get a batch of validation data
    val_batch = next(iter(val_dataloader))
    images, targets = val_batch
    
    # Move PyTorch model to CPU for fair comparison with ONNX
    model = model.to('cpu')
    images = images.to('cpu')
    targets = targets.to('cpu')
    
    # Get PyTorch predictions
    model.eval()
    with torch.no_grad():
        pytorch_predictions = model(images)
    
    # Initialize ONNX Runtime session
    ort_session = onnxruntime.InferenceSession(onnx_path)
    input_name = ort_session.get_inputs()[0].name
    
    # Get ONNX predictions
    onnx_inputs = {input_name: images.numpy()}
    onnx_outputs = ort_session.run(None, onnx_inputs)
    onnx_predictions = torch.from_numpy(onnx_outputs[0])
    
    # Display examples side by side
    batch_size = images.shape[0]
    num_examples = min(num_examples, batch_size)
    
    for i in range(num_examples):
        # Plot the comparison
        fig, axes = compare_pytorch_onnx_side_by_side(
            image=images[i],
            ground_truth=targets[i],
            pytorch_pred=pytorch_predictions[i],
            onnx_pred=onnx_predictions[i],
            grid_lines=grid_lines,
            example_num=i+1
        )
        plt.show()
        
        # Print numerical differences
        pt_gt_diff = torch.max(torch.abs(pytorch_predictions[i] - targets[i])).item()
        onnx_gt_diff = torch.max(torch.abs(onnx_predictions[i] - targets[i])).item()
        pt_onnx_diff = torch.max(torch.abs(pytorch_predictions[i] - onnx_predictions[i])).item()
        
        print(f"Example {i+1} - Maximum absolute differences:")
        print(f"  PyTorch vs Ground Truth: {pt_gt_diff:.6f}")
        print(f"  ONNX vs Ground Truth: {onnx_gt_diff:.6f}")
        print(f"  PyTorch vs ONNX: {pt_onnx_diff:.6f}")

# Run the comparison
run_pytorch_onnx_comparison(
    model=model, 
    onnx_path="./object_detection.onnx", 
    data_module=data_module,
    num_examples=3
)

In [ ]:
def compare_models_on_dataset(model, onnx_path, data_module, difference_threshold=0.01):
    """
    Compare PyTorch and ONNX models on the entire validation dataset and 
    find examples where they differ significantly
    
    Args:
        model: The PyTorch model
        onnx_path: Path to the ONNX model
        data_module: PyTorch Lightning data module
        difference_threshold: Threshold to consider a difference significant
    
    Returns:
        list: List of tuples (image, ground_truth, pytorch_pred, onnx_pred, max_diff)
              for examples where the difference exceeds the threshold
    """
    # Ensure setup is called
    data_module.setup()
    val_dataloader = data_module.val_dataloader()
    
    # Move PyTorch model to CPU for fair comparison with ONNX
    model = model.to('cpu')
    model.eval()
    
    # Initialize ONNX Runtime session
    ort_session = onnxruntime.InferenceSession(onnx_path)
    input_name = ort_session.get_inputs()[0].name
    
    # List to store examples with significant differences
    significant_differences = []
    
    # Statistics
    total_examples = 0
    total_correct_match = 0
    difference_histogram = []
    
    # Process all batches
    print("Processing validation dataset...")
    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(val_dataloader):
            # Move tensors to CPU
            images = images.to('cpu')
            targets = targets.to('cpu')
            
            # Get PyTorch predictions
            pytorch_predictions = model(images)
            
            # Get ONNX predictions
            onnx_inputs = {input_name: images.numpy()}
            onnx_outputs = ort_session.run(None, onnx_inputs)
            onnx_predictions = torch.from_numpy(onnx_outputs[0])
            
            # Compare predictions for each example in the batch
            for i in range(images.shape[0]):
                total_examples += 1
                
                # Calculate max absolute difference
                max_diff = torch.max(torch.abs(pytorch_predictions[i] - onnx_predictions[i])).item()
                difference_histogram.append(max_diff)
                
                # Check if predictions match exactly (after thresholding)
                threshold = 0.5
                pytorch_classes = (pytorch_predictions[i] > threshold).int()
                onnx_classes = (onnx_predictions[i] > threshold).int()
                
                exact_match = torch.all(pytorch_classes == onnx_classes).item()
                if exact_match:
                    total_correct_match += 1
                
                # If difference exceeds threshold, save this example
                if max_diff > difference_threshold or not exact_match:
                    significant_differences.append((
                        images[i].clone(),
                        targets[i].clone(),
                        pytorch_predictions[i].clone(),
                        onnx_predictions[i].clone(),
                        max_diff,
                        exact_match
                    ))
            
            # Print progress every few batches
            if (batch_idx + 1) % 5 == 0 or batch_idx == len(val_dataloader) - 1:
                print(f"Processed {batch_idx+1}/{len(val_dataloader)} batches, "
                      f"found {len(significant_differences)} examples with significant differences")
    
    # Print summary statistics
    print("\nValidation Dataset Summary:")
    print(f"Total examples processed: {total_examples}")
    print(f"Examples with exact class match: {total_correct_match} ({100 * total_correct_match / total_examples:.2f}%)")
    print(f"Examples with significant differences: {len(significant_differences)}")
    
    # Calculate and print difference statistics
    if difference_histogram:
        avg_diff = sum(difference_histogram) / len(difference_histogram)
        max_diff = max(difference_histogram)
        print(f"Average max difference: {avg_diff:.6f}")
        print(f"Maximum difference: {max_diff:.6f}")
    
    return significant_differences

# Run the comparison
significant_differences = compare_models_on_dataset(
    model=model, 
    onnx_path="./light_object_detection.onnx", 
    data_module=data_module,
    difference_threshold=0.0001  # Adjust this threshold as needed
)

print(f"\nFound {len(significant_differences)} examples with significant differences")

In [ ]:
import onnx
import numpy as np
import sys
import os

def analyze_onnx_model(model_path):
    """
    Analyze an ONNX model and print detailed information
    
    Args:
        model_path: Path to the ONNX model file
    """
    print(f"Analyzing ONNX model: {model_path}")
    print("-" * 80)
    
    # Load the ONNX model
    model = onnx.load(model_path)
    
    # Check model validity
    try:
        onnx.checker.check_model(model)
        print("✓ Model is valid")
    except Exception as e:
        print(f"✗ Model is invalid: {e}")
    
    # Basic model info
    print("\n== Model Info ==")
    print(f"IR Version: {model.ir_version}")
    print(f"Producer: {model.producer_name} {model.producer_version}")
    print(f"Domain: {model.domain}")
    print(f"Model Version: {model.model_version}")
    print(f"Doc: {model.doc_string}")
    
    # Count nodes by type
    node_types = {}
    for node in model.graph.node:
        if node.op_type in node_types:
            node_types[node.op_type] += 1
        else:
            node_types[node.op_type] = 1
    
    print("\n== Node Types ==")
    for op_type, count in sorted(node_types.items()):
        print(f"{op_type}: {count}")
    print(f"Total nodes: {len(model.graph.node)}")
    
    # Analyze inputs
    print("\n== Inputs ==")
    for i, input_data in enumerate(model.graph.input):
        name = input_data.name
        shape = []
        for dim in input_data.type.tensor_type.shape.dim:
            if dim.dim_param:
                shape.append(dim.dim_param)
            else:
                shape.append(dim.dim_value)
        print(f"Input #{i}: {name}, Shape: {shape}")
    
    # Analyze outputs
    print("\n== Outputs ==")
    for i, output in enumerate(model.graph.output):
        name = output.name
        shape = []
        for dim in output.type.tensor_type.shape.dim:
            if dim.dim_param:
                shape.append(dim.dim_param)
            else:
                shape.append(dim.dim_value)
        print(f"Output #{i}: {name}, Shape: {shape}")
    
    # Analyze initializers (weights and biases)
    print("\n== Initializers (Parameters) ==")
    total_params = 0
    total_size_bytes = 0
    
    for i, initializer in enumerate(model.graph.initializer):
        shape = [dim for dim in initializer.dims]
        num_params = np.prod(shape)
        total_params += num_params
        
        # Calculate memory size
        elem_type = initializer.data_type
        bytes_per_element = 4  # Default for float32
        if elem_type == 1:  # FLOAT
            bytes_per_element = 4
        elif elem_type == 2:  # UINT8
            bytes_per_element = 1
        elif elem_type == 3:  # INT8
            bytes_per_element = 1
        elif elem_type == 5:  # INT16
            bytes_per_element = 2
        elif elem_type == 6:  # INT32
            bytes_per_element = 4
        elif elem_type == 7:  # INT64
            bytes_per_element = 8
        elif elem_type == 10:  # FLOAT16
            bytes_per_element = 2
        elif elem_type == 11:  # DOUBLE
            bytes_per_element = 8
            
        param_size_bytes = num_params * bytes_per_element
        total_size_bytes += param_size_bytes
        
        print(f"Initializer #{i}: {initializer.name}")
        print(f"  Shape: {shape}, Elements: {num_params}, Size: {format_bytes(param_size_bytes)}")
    
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Total parameter size: {format_bytes(total_size_bytes)}")
    
    # Estimate C file size (parameters + overhead)
    estimated_c_file_size = total_size_bytes * 7  # Rough estimate based on float representation in C
    print(f"Estimated C file size: {format_bytes(estimated_c_file_size)} (rough estimate)")
    
    print("\n== Memory Analysis ==")
    print(f"Parameter memory: {format_bytes(total_size_bytes)}")
    
    # Estimate activation memory based on intermediate tensors
    # This is a rough estimation
    activation_memory = 0
    for node in model.graph.node:
        for output in node.output:
            # Find corresponding value info if available
            for value_info in model.graph.value_info:
                if value_info.name == output:
                    shape = [dim.dim_value for dim in value_info.type.tensor_type.shape.dim]
                    if all(shape):  # Only count if we have complete shape info
                        activation_memory += np.prod(shape) * 4  # Assuming float32
    
    print(f"Activation memory (estimated): {format_bytes(activation_memory)}")
    print(f"Total memory footprint (estimated): {format_bytes(total_size_bytes + activation_memory)}")

def format_bytes(size):
    """Format bytes to human-readable string"""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size < 1024.0 or unit == 'GB':
            return f"{size:.2f} {unit}"
        size /= 1024.0

analyze_onnx_model('light_object_detection.onnx')